# Healpix Sphere Visualization at Multiple nside Resolutions

Visualises a unit sphere with healpix pixel **edges** for each nside level.
Based on: https://github.com/Python-World/snippets/blob/master/sphere_healpy.md

In [ ]:
import numpy as np
import healpy as hp

## Configuration

In [ ]:
# Sphere parameters
R = 1.0
nside_list = [256, 128, 64, 32, 16, 8, 4, 2, 1]
grid_res = 101  # sphere mesh resolution
ordering = 'nested'  # Use nested HEALPix ordering

## Helper: generate smooth sphere mesh

In [ ]:
def make_sphere(r=1.0, res=101):
    """Return (x, y, z) arrays for a smooth unit sphere."""
    phi, theta = np.mgrid[0:np.pi:res*1j, 0:2*np.pi:res*1j]
    x = r * np.sin(phi) * np.cos(theta)
    y = r * np.sin(phi) * np.sin(theta)
    z = r * np.cos(phi)
    return x, y, z

## Plot all nside levels using Matplotlib (2D projection)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D

ncols = 3
nrows = (len(nside_list) + ncols - 1) // ncols

fig = plt.figure(figsize=(20, 7 * nrows))

for idx, nside in enumerate(nside_list):
    ax = fig.add_subplot(nrows, ncols, idx + 1, projection='3d')
    
    npix = hp.nside2npix(nside)
    
    # Smooth sphere mesh
    x, y, z = make_sphere(R, grid_res)
    ax.plot_surface(x, y, z, color='cyan', alpha=0.25, rstride=4, cstride=4)
    
    # Pixel edges via hp.boundaries() with nested ordering and coloring
    step = 1 if nside <= 32 else (max(1, nside // 8))
    pixels = np.arange(0, npix, step)
    cmap = cm.get_cmap('tab20c')  # Use a colormap
    
    for pix in pixels:
        bnd = hp.boundaries(nside, pix, nest=True)   # shape (3, 4)
        color = cmap((pix % 20) / 20)  # Cycle through colors for visibility
        ax.plot3D(
            np.append(bnd[0], np.nan),
            np.append(bnd[1], np.nan),
            np.append(bnd[2], np.nan),
            color=color, linewidth=0.5,
        )
    
    # Plot pixel centers as dots for nside <= 128
    if nside <= 128:
        theta, phi = hp.pix2ang(nside, pixels, nest=True)
        x_pix = np.sin(theta) * np.cos(phi)
        y_pix = np.sin(theta) * np.sin(phi)
        z_pix = np.cos(theta)
        ax.scatter(x_pix, y_pix, z_pix, s=10, c='red', alpha=0.5)
    
    ax.set_title(f"nside={nside}, npix={npix}, nested", fontsize=10)
    ax.set_box_aspect([1, 1, 1])

plt.tight_layout()
plt.savefig("healpy_spheres_all.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved healpy_spheres_all.png")

## Per-nside individual plots (large)

In [ ]:
for nside in nside_list:
    npix = hp.nside2npix(nside)
    
    x, y, z = make_sphere(R, grid_res)
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(x, y, z, color='cyan', alpha=0.2)
    
    # Pixel edges with nested ordering and coloring
    step = 1 if nside <= 32 else (max(1, nside // 8))
    pixels = np.arange(0, npix, step)
    cmap = cm.get_cmap('hsv')
    
    for pix in pixels:
        bnd = hp.boundaries(nside, pix, nest=True)   # shape (3, 4)
        color = cmap(pix / npix)
        ax.plot3D(
            np.append(bnd[0], np.nan),
            np.append(bnd[1], np.nan),
            np.append(bnd[2], np.nan),
            color=color, linewidth=0.7,
        )
    
    # Plot pixel centers as dots for nside <= 128
    if nside <= 128:
        theta, phi = hp.pix2ang(nside, pixels, nest=True)
        x_pix = np.sin(theta) * np.cos(phi)
        y_pix = np.sin(theta) * np.sin(phi)
        z_pix = np.cos(theta)
        ax.scatter(x_pix, y_pix, z_pix, s=20, c='red', alpha=0.6)
    
    ax.set_title(f"nside={nside}, npix={npix}, nested")
    ax.set_box_aspect([1, 1, 1])
    
    fname = f"healpy_sphere_nside{nside}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {fname}")

## Pixel area comparison

In [ ]:
for nside in [128, 64, 32, 16]:
    NPIX = hp.nside2npix(nside)

    # Create a map with a Gaussian bump at the north pole
    m = np.zeros(NPIX)
    m[0] = 1.0  # Set north pole pixel
    m_smoothed = hp.smoothing(m, fwhm=np.radians(10), nest=True)

    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Get colormap for smoothed values
    cmap = plt.cm.get_cmap('hot')
    norm = plt.Normalize(vmin=m_smoothed.min(), vmax=m_smoothed.max())
    
    # Plot all pixels
    for pix in range(NPIX):
        # Get pixel boundaries (vertices on unit sphere)
        bnd = hp.boundaries(nside, pix, nest=True)  # shape (3, 4)
        
        # Extract x, y, z coordinates
        x = bnd[0]
        y = bnd[1]
        z = bnd[2]
        
        # Create quad vertices for 3D plotting
        vertices = np.column_stack([x, y, z])
        
        # Get color based on smoothed value
        color = cmap(norm(m_smoothed[pix]))
        
        # Plot as filled 3D polygon
        from mpl_toolkits.mplot3d.art3d import Poly3DCollection
        poly = [[vertices[0], vertices[1], vertices[2], vertices[3]]]
        ax.add_collection3d(Poly3DCollection(poly, alpha=0.95, facecolor=color, edgecolor='gray', linewidth=0.1))
    
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f"3D Gaussian on Sphere (nside={nside}, npix={NPIX})")
    ax.set_box_aspect([1, 1, 1])
    
    fname = f"healpy_3d_gaussian_nside{nside}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {fname}")

In [ ]:
for nside in [256, 128, 64, 32]:
    NPIX = hp.nside2npix(nside)

    # Create a map with a Gaussian bump at the north pole
    m = np.zeros(NPIX)
    m[0] = 1.0  # Set north pole pixel
    m_smoothed = hp.smoothing(m, fwhm=np.radians(10), nest=True)

    plt.figure(figsize=(12, 8))
    hp.mollview(m_smoothed, nest=True, title=f"Smoothed Gaussian (nside={nside})", cmap='hot')

    fname = f"healpy_mollview_gaussian_nside{nside}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {fname}")

## Mollview with Gaussian Blur

In [ ]:
for nside in nside_list:
    NPIX = hp.nside2npix(nside)
    m = np.arange(NPIX)

    plt.figure(figsize=(12, 8))
    hp.mollview(m, nest=True, title=f"Mollview image NESTED (nside={nside}, npix={NPIX})", cmap='viridis')

    fname = f"healpy_mollview_nside{nside}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {fname}")

## Mollview Projections for each nside